# SKEMPI Dual-Protein Cross-Attention Model — Task 2 (v2)

**v2 changes, in response to v1 results (R² ≈ 0, F1 = 0 — essentially no learned signal):**
1. **Local mutation-site pooling** — v1 only mean-pooled the *entire* chain, which dilutes a single-residue change across hundreds of positions. v2 additionally extracts the token embedding right at the mutated residue(s) and computes a local diff, alongside the global diff.
2. **Standardized ΔΔG target** — v1's raw ΔΔG (std ≈ 1.7) dominated the loss over BCE (~0.7), skewing gradient signal toward the regression term. v2 z-scores the target using train statistics.
3. **Differential learning rates** — v1 used one LR (1e-5) for everything, too conservative for the freshly-initialized head. v2 gives the head/cross-attention/local-pooling components a higher LR, keeps the small set of unfrozen ESM-2 layers on a lower LR.
4. **Explicit baseline printout** — prints "always predict the mean / majority class" performance before training, so you have an honest floor to compare against rather than eyeballing whether R²=-0.03 is bad.

Same overall design as v1 otherwise: siamese wt-vs-mut encoding through shared cross-attention, multi-task regression + classification head.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q transformers scipy scikit-learn

import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from scipy.stats import pearsonr
from sklearn.metrics import r2_score, mean_squared_error, roc_auc_score, f1_score, accuracy_score
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

DATA_DIR = '/content/drive/MyDrive/drug_discovery_project'
CKPT_DIR = '/content/drive/MyDrive/drug_discovery_project/checkpoints'
import os
os.makedirs(CKPT_DIR, exist_ok=True)

Mounted at /content/drive
Using device: cuda


## 2. Config

In [ ]:
MODEL_NAME = 'facebook/esm2_t12_35M_UR50D'
MAX_LEN = 512
MAX_MUTATIONS = 6       # slots for local mutation-site pooling; rows with more mutations still get global pooling for the rest
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
HEAD_LR = 3e-4           # head / cross-attention / pooling — randomly initialized, needs to move faster
ENCODER_LR = 2e-5        # unfrozen ESM-2 layers — pretrained, keep conservative
WEIGHT_DECAY = 0.01
EPOCHS = 20
PATIENCE = 5
FREEZE_UP_TO_LAYER = 10   # ESM2-t12 has 12 layers (0-11); freeze embeddings + layers 0-9
DDG_LOSS_WEIGHT = 1.0
CLS_LOSS_WEIGHT = 1.0
DROPOUT = 0.3
N_ATTN_HEADS = 4

## 3. Load data and compute normalization / baselines

Compute train-set ΔΔG mean/std for target standardization, and print trivial baselines (predict train mean; predict majority class) so later results have an honest floor to compare against.

In [ ]:
train_df = pd.read_csv(f'{DATA_DIR}/skempi_train_final.csv')
val_df = pd.read_csv(f'{DATA_DIR}/skempi_val_final.csv')
test_df = pd.read_csv(f'{DATA_DIR}/skempi_test_final.csv')

print(f"Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}")

TRAIN_DDG_MEAN = train_df['ddG_kcal_mol'].mean()
TRAIN_DDG_STD = train_df['ddG_kcal_mol'].std()
print(f"\nTrain ddG mean: {TRAIN_DDG_MEAN:.4f}  std: {TRAIN_DDG_STD:.4f}")

majority_class = train_df['resistant_label'].mode()[0]
print(f"Majority class (train): {majority_class}")

print("\n--- Trivial baselines (for comparison against model results) ---")
for name, df in [('val', val_df), ('test', test_df)]:
    mean_pred = np.full(len(df), TRAIN_DDG_MEAN)
    r2 = r2_score(df['ddG_kcal_mol'], mean_pred)
    rmse = np.sqrt(mean_squared_error(df['ddG_kcal_mol'], mean_pred))
    maj_pred = np.full(len(df), majority_class)
    acc = accuracy_score(df['resistant_label'], maj_pred)
    f1 = f1_score(df['resistant_label'], maj_pred, zero_division=0)
    print(f"{name}: predict-train-mean R2={r2:.4f} RMSE={rmse:.4f}  |  predict-majority-class acc={acc:.4f} f1={f1:.4f}")
print("Any model worth keeping should clearly beat these numbers, not just approach them.")

Train: 1182  Val: 192  Test: 288

Train ddG mean: 1.0598  std: 1.6986
Majority class (train): 0

--- Trivial baselines (for comparison against model results) ---
val: predict-train-mean R2=-0.0191 RMSE=2.0055  |  predict-majority-class acc=0.4844 f1=0.0000
test: predict-train-mean R2=-0.0322 RMSE=1.6092  |  predict-majority-class acc=0.6181 f1=0.0000
Any model worth keeping should clearly beat these numbers, not just approach them.


## 4. Dataset & tokenization

Adds mutation position parsing on top of v1's 4-sequence tokenization. `Mutation(s)_cleaned` entries were already validated against the fetched sequence during preprocessing (only `mutant_seq_ok==True` rows made it into these files), so position-1 indexing is trustworthy here. ESM-2's tokenizer prepends a `<cls>` token, so a 1-indexed residue position `p` lands at token index `p` (0-indexed AA position `p-1`, plus 1 for `<cls>`).

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MUT_RE = re.compile(r'^([A-Za-z])([A-Za-z])(\d+)([A-Za-z])$')

def parse_mutation_positions(mut_str, chains_side1, chains_side2, max_len):
    """Returns (pos_a, mask_a, pos_b, mask_b), each length MAX_MUTATIONS."""
    pos_a = [0] * MAX_MUTATIONS
    mask_a = [0.0] * MAX_MUTATIONS
    pos_b = [0] * MAX_MUTATIONS
    mask_b = [0.0] * MAX_MUTATIONS
    ia, ib = 0, 0
    for m in str(mut_str).split(','):
        match = MUT_RE.match(m.strip())
        if not match:
            continue
        _, chain, pos_str, _ = match.groups()
        pos = int(pos_str)  # 1-indexed AA position -> token index (accounts for <cls> at index 0)
        if pos < 1 or pos > max_len - 2:  # leave room for <eos>; drop if truncated out
            continue
        if chain in str(chains_side1) and ia < MAX_MUTATIONS:
            pos_a[ia] = pos
            mask_a[ia] = 1.0
            ia += 1
        elif chain in str(chains_side2) and ib < MAX_MUTATIONS:
            pos_b[ib] = pos
            mask_b[ib] = 1.0
            ib += 1
    return pos_a, mask_a, pos_b, mask_b


class SkempiDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=MAX_LEN, ddg_mean=TRAIN_DDG_MEAN, ddg_std=TRAIN_DDG_STD):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.ddg_mean = ddg_mean
        self.ddg_std = ddg_std

    def __len__(self):
        return len(self.df)

    def _tok(self, seq):
        seq = str(seq) if pd.notna(seq) else ''
        enc = self.tokenizer(
            seq, truncation=True, max_length=self.max_len,
            padding='max_length', return_tensors='pt'
        )
        return enc['input_ids'].squeeze(0), enc['attention_mask'].squeeze(0)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        a_wt_ids, a_wt_mask = self._tok(row['seq1_wt'])
        b_wt_ids, b_wt_mask = self._tok(row['seq2_wt'])
        a_mut_ids, a_mut_mask = self._tok(row['seq1_mut'])
        b_mut_ids, b_mut_mask = self._tok(row['seq2_mut'])

        pos_a, mask_a, pos_b, mask_b = parse_mutation_positions(
            row['Mutation(s)_cleaned'], row['chains_side1'], row['chains_side2'], self.max_len
        )

        raw_ddg = row['ddG_kcal_mol']
        norm_ddg = (raw_ddg - self.ddg_mean) / self.ddg_std

        return {
            'a_wt_ids': a_wt_ids, 'a_wt_mask': a_wt_mask,
            'b_wt_ids': b_wt_ids, 'b_wt_mask': b_wt_mask,
            'a_mut_ids': a_mut_ids, 'a_mut_mask': a_mut_mask,
            'b_mut_ids': b_mut_ids, 'b_mut_mask': b_mut_mask,
            'pos_a': torch.tensor(pos_a, dtype=torch.long),
            'mask_a': torch.tensor(mask_a, dtype=torch.float32),
            'pos_b': torch.tensor(pos_b, dtype=torch.long),
            'mask_b': torch.tensor(mask_b, dtype=torch.float32),
            'ddg_norm': torch.tensor(norm_ddg, dtype=torch.float32),
            'ddg_raw': torch.tensor(raw_ddg, dtype=torch.float32),
            'label': torch.tensor(row['resistant_label'], dtype=torch.float32),
        }

train_ds = SkempiDataset(train_df, tokenizer)
val_ds = SkempiDataset(val_df, tokenizer)
test_ds = SkempiDataset(test_df, tokenizer)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# sanity check: how many rows have at least one local mutation position captured?
n_with_local = sum(1 for i in range(len(train_ds)) if train_ds[i]['mask_a'].sum() + train_ds[i]['mask_b'].sum() > 0)
print(f"Train rows with >=1 local mutation site captured: {n_with_local} / {len(train_ds)}")
print(f"Batches per epoch — train: {len(train_loader)}, val: {len(val_loader)}, test: {len(test_loader)}")

config.json:   0%|          | 0.00/778 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Train rows with >=1 local mutation site captured: 1181 / 1182
Batches per epoch — train: 296, val: 48, test: 72


## 5. Model

`encode_complex` now returns both the global pooled embedding *and* the raw fused token sequences, so `gather_local_mean` can pull out embeddings at the exact mutated positions. Local and global diffs are concatenated together into the final feature vector.

In [ ]:
class CrossAttentionBlock(nn.Module):
    """Query sequence attends over Key/Value sequence, with padding masking."""
    def __init__(self, hidden_dim, n_heads=N_ATTN_HEADS, dropout=DROPOUT):
        super().__init__()
        self.attn = nn.MultiheadAttention(hidden_dim, n_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, query, query_mask, kv, kv_mask):
        kv_key_padding_mask = (kv_mask == 0)
        attn_out, _ = self.attn(query, kv, kv, key_padding_mask=kv_key_padding_mask)
        fused = self.norm(query + attn_out)
        return fused


def masked_mean_pool(x, mask):
    mask = mask.unsqueeze(-1).float()
    summed = (x * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-6)
    return summed / counts


def gather_local_mean(x, positions, valid_mask):
    """x: (batch, seq_len, hidden). positions/valid_mask: (batch, MAX_MUTATIONS).
    Returns (batch, hidden) mean of token embeddings at the given positions, zeros if none valid."""
    batch, seq_len, hidden = x.shape
    idx = positions.clamp(0, seq_len - 1).unsqueeze(-1).expand(-1, -1, hidden)  # (batch, K, hidden)
    gathered = torch.gather(x, 1, idx)  # (batch, K, hidden)
    mask = valid_mask.unsqueeze(-1)  # (batch, K, 1)
    summed = (gathered * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-6)
    return summed / counts


class SkempiCrossAttentionModel(nn.Module):
    def __init__(self, model_name=MODEL_NAME, freeze_up_to_layer=FREEZE_UP_TO_LAYER, dropout=DROPOUT):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_dim = self.encoder.config.hidden_size

        for param in self.encoder.embeddings.parameters():
            param.requires_grad = False
        for i, layer in enumerate(self.encoder.encoder.layer):
            if i < freeze_up_to_layer:
                for param in layer.parameters():
                    param.requires_grad = False

        self.cross_a_to_b = CrossAttentionBlock(hidden_dim)
        self.cross_b_to_a = CrossAttentionBlock(hidden_dim)

        complex_dim = hidden_dim * 2       # pooled(A attends B) concat pooled(B attends A)
        # per complex: [global, local] -> hidden*2 + hidden*2 = hidden*4
        full_complex_dim = complex_dim * 2
        combined_dim = full_complex_dim * 3  # [wt, mut, diff]

        self.head = nn.Sequential(
            nn.Linear(combined_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.ddg_head = nn.Linear(64, 1)
        self.cls_head = nn.Linear(64, 1)

    def encode_complex(self, a_ids, a_mask, b_ids, b_mask):
        a_tokens = self.encoder(input_ids=a_ids, attention_mask=a_mask).last_hidden_state
        b_tokens = self.encoder(input_ids=b_ids, attention_mask=b_mask).last_hidden_state

        fused_a = self.cross_a_to_b(a_tokens, a_mask, b_tokens, b_mask)
        fused_b = self.cross_b_to_a(b_tokens, b_mask, a_tokens, a_mask)

        pooled_global = torch.cat([masked_mean_pool(fused_a, a_mask), masked_mean_pool(fused_b, b_mask)], dim=-1)
        return pooled_global, fused_a, fused_b

    def full_complex_vector(self, a_ids, a_mask, b_ids, b_mask, pos_a, mask_a, pos_b, mask_b):
        pooled_global, fused_a, fused_b = self.encode_complex(a_ids, a_mask, b_ids, b_mask)
        local_a = gather_local_mean(fused_a, pos_a, mask_a)
        local_b = gather_local_mean(fused_b, pos_b, mask_b)
        pooled_local = torch.cat([local_a, local_b], dim=-1)
        return torch.cat([pooled_global, pooled_local], dim=-1)  # (batch, hidden*4)

    def forward(self, batch):
        wt_vec = self.full_complex_vector(
            batch['a_wt_ids'], batch['a_wt_mask'], batch['b_wt_ids'], batch['b_wt_mask'],
            batch['pos_a'], batch['mask_a'], batch['pos_b'], batch['mask_b'],
        )
        mut_vec = self.full_complex_vector(
            batch['a_mut_ids'], batch['a_mut_mask'], batch['b_mut_ids'], batch['b_mut_mask'],
            batch['pos_a'], batch['mask_a'], batch['pos_b'], batch['mask_b'],
        )
        diff_vec = mut_vec - wt_vec

        combined = torch.cat([wt_vec, mut_vec, diff_vec], dim=-1)
        features = self.head(combined)

        ddg_pred_norm = self.ddg_head(features).squeeze(-1)
        cls_logit = self.cls_head(features).squeeze(-1)
        return ddg_pred_norm, cls_logit


model = SkempiCrossAttentionModel().to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({trainable/total:.1%})")

model.safetensors: reconstructing file:   0%|          |  0.00B /  136MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t12_35M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable params: 9,114,515 / 36,840,755 (24.7%)


## 6. Training loop

Two param groups with different learning rates: pretrained-but-unfrozen ESM-2 layers on `ENCODER_LR`, everything newly initialized (cross-attention, pooling, head) on the higher `HEAD_LR`.

In [ ]:
encoder_params, head_params = [], []
for name, p in model.named_parameters():
    if not p.requires_grad:
        continue
    if name.startswith('encoder.'):
        encoder_params.append(p)
    else:
        head_params.append(p)

optimizer = torch.optim.AdamW([
    {'params': encoder_params, 'lr': ENCODER_LR},
    {'params': head_params, 'lr': HEAD_LR},
], weight_decay=WEIGHT_DECAY)

total_steps = (len(train_loader) // GRAD_ACCUM_STEPS) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
)

mse_loss = nn.MSELoss()
bce_loss = nn.BCEWithLogitsLoss()

def move_batch(batch, device):
    return {k: v.to(device) for k, v in batch.items()}

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0
    optimizer.zero_grad()

    with torch.set_grad_enabled(train):
        for step, batch in enumerate(loader):
            batch = move_batch(batch, DEVICE)
            ddg_pred_norm, cls_logit = model(batch)

            loss = DDG_LOSS_WEIGHT * mse_loss(ddg_pred_norm, batch['ddg_norm']) + \
                   CLS_LOSS_WEIGHT * bce_loss(cls_logit, batch['label'])

            if train:
                (loss / GRAD_ACCUM_STEPS).backward()
                if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(loader):
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()

            total_loss += loss.item()

    return total_loss / len(loader)


best_val_loss = float('inf')
patience_counter = 0
history = {'train_loss': [], 'val_loss': []}

for epoch in range(EPOCHS):
    train_loss = run_epoch(train_loader, train=True)
    val_loss = run_epoch(val_loader, train=False)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)

    print(f"Epoch {epoch+1}/{EPOCHS} — train_loss: {train_loss:.4f}  val_loss: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), f'{CKPT_DIR}/skempi_crossattn_v2_best.pt')
        print("  -> saved new best checkpoint")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1} (no val improvement for {PATIENCE} epochs)")
            break

Epoch 1/20 — train_loss: 1.6588  val_loss: 2.1609
  -> saved new best checkpoint
Epoch 2/20 — train_loss: 1.5228  val_loss: 2.0346
  -> saved new best checkpoint
Epoch 3/20 — train_loss: 1.3415  val_loss: 1.8256
  -> saved new best checkpoint
Epoch 4/20 — train_loss: 1.1730  val_loss: 1.9259
Epoch 5/20 — train_loss: 1.0148  val_loss: 1.8814
Epoch 6/20 — train_loss: 0.9544  val_loss: 1.8948
Epoch 7/20 — train_loss: 0.8475  val_loss: 2.1618
Epoch 8/20 — train_loss: 0.7540  val_loss: 2.0767
Early stopping at epoch 8 (no val improvement for 5 epochs)


## 7. Evaluation

Un-normalizes ΔΔG predictions before computing R²/RMSE so units are back in kcal/mol. Prints the trivial-baseline numbers again alongside for direct comparison.

In [ ]:
def evaluate(loader, name):
    model.eval()
    model.load_state_dict(torch.load(f'{CKPT_DIR}/skempi_crossattn_v2_best.pt'))

    all_ddg_true, all_ddg_pred = [], []
    all_cls_true, all_cls_prob = [], []

    with torch.no_grad():
        for batch in loader:
            batch_dev = move_batch(batch, DEVICE)
            ddg_pred_norm, cls_logit = model(batch_dev)
            ddg_pred = ddg_pred_norm.cpu().numpy() * TRAIN_DDG_STD + TRAIN_DDG_MEAN

            all_ddg_true.extend(batch['ddg_raw'].numpy())
            all_ddg_pred.extend(ddg_pred)
            all_cls_true.extend(batch['label'].numpy())
            all_cls_prob.extend(torch.sigmoid(cls_logit).cpu().numpy())

    all_ddg_true = np.array(all_ddg_true)
    all_ddg_pred = np.array(all_ddg_pred)
    all_cls_true = np.array(all_cls_true)
    all_cls_prob = np.array(all_cls_prob)
    all_cls_pred = (all_cls_prob >= 0.5).astype(int)

    r2 = r2_score(all_ddg_true, all_ddg_pred)
    pearson_r, _ = pearsonr(all_ddg_true, all_ddg_pred)
    rmse = np.sqrt(mean_squared_error(all_ddg_true, all_ddg_pred))
    auc = roc_auc_score(all_cls_true, all_cls_prob)
    f1 = f1_score(all_cls_true, all_cls_pred, zero_division=0)
    acc = accuracy_score(all_cls_true, all_cls_pred)

    baseline_r2 = r2_score(all_ddg_true, np.full(len(all_ddg_true), TRAIN_DDG_MEAN))

    print(f"=== {name} ===")
    print(f"Regression (ΔΔG):  R²: {r2:.4f} (baseline: {baseline_r2:.4f})   Pearson r: {pearson_r:.4f}   RMSE: {rmse:.4f}")
    print(f"Classification:   AUC: {auc:.4f}   F1: {f1:.4f}   Accuracy: {acc:.4f}")

    return {'r2': r2, 'pearson_r': pearson_r, 'rmse': rmse, 'auc': auc, 'f1': f1, 'accuracy': acc}

val_metrics = evaluate(val_loader, 'Validation')
print()
test_metrics = evaluate(test_loader, 'Test')

=== Validation ===
Regression (ΔΔG):  R²: 0.1484 (baseline: -0.0191)   Pearson r: 0.4299   RMSE: 1.8332
Classification:   AUC: 0.6649   F1: 0.5775   Accuracy: 0.5885

=== Test ===
Regression (ΔΔG):  R²: 0.1064 (baseline: -0.0322)   Pearson r: 0.3720   RMSE: 1.4973
Classification:   AUC: 0.6931   F1: 0.5952   Accuracy: 0.6458


## 8. If this still doesn't beat baseline

In rough order of likely payoff:

1. **Check `n_with_local` from section 4.** If it's a small fraction of rows, most mutation positions are getting truncated out by `MAX_LEN=512` — either raise `MAX_LEN` (memory cost) or filter to rows where the mutation site survives truncation.
2. **Try local-only features first** (drop the global pooled terms from `combined` temporarily) to isolate whether the local signal alone is more learnable — if it is, the global term may be adding noise the small dataset can't average out.
3. **Increase `HEAD_LR` further** (e.g. 1e-3) or fully freeze the encoder (`FREEZE_UP_TO_LAYER=12`) so it's a pure fixed-feature-extractor setup — with this little data, the safest regime is often "don't fine-tune the backbone at all."
4. **Grouped k-fold CV** (by `complex_id`) to check whether a single val split is just an unlucky sample — with 52 val complexes, split variance alone could be masking real signal.
5. **Sanity-check the target itself** — plot ΔΔG vs. a cheap classical feature (e.g. a BLOSUM62 substitution score at the mutation site) to confirm the signal is learnable at all before assuming the architecture is at fault.